# Kriging ordinario, sensibilidad y validación

Ejemplo sintético para distinguir estructura espacial, error de observación, pesos e incertidumbre.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

coords = np.array([[0,0],[2,.5],[4.5,.2],[.8,2.4],[3.1,2],[5,2.8],[1.4,4.5],[4.2,4.6]], float)
z = np.array([182,188,199,179,194,207,185,203], float)
u0 = np.array([2.7,3.1])

def cov_exp(a, b, alcance=2.7, var_estructura=64., var_error=0., misma_muestra=False):
    a, b = np.atleast_2d(a), np.atleast_2d(b)
    d = np.linalg.norm(a[:,None,:]-b[None,:,:], axis=2)
    C = var_estructura*np.exp(-d/alcance)
    if misma_muestra: C += np.eye(len(a))*var_error
    return C

def ok(coords, z, objetivo, alcance=2.7, var_estructura=64., var_error=4.):
    C = cov_exp(coords, coords, alcance, var_estructura, var_error, True)
    c0 = cov_exp(coords, objetivo, alcance, var_estructura).ravel()
    A = np.block([[C, np.ones((len(coords),1))], [np.ones((1,len(coords))), np.zeros((1,1))]])
    lam_mu = np.linalg.solve(A, np.r_[c0, 1.])
    lam, mu = lam_mu[:-1], lam_mu[-1]
    return lam@z, max(var_estructura-lam@c0-mu, 0.), lam

In [ ]:
escenarios = [('corto', 1.3, 4), ('referencia', 2.7, 4), ('más error', 2.7, 20)]
filas = []
for nombre, alcance, error in escenarios:
    pred, var, lam = ok(coords, z, u0, alcance=alcance, var_error=error)
    filas.append({'escenario': nombre, 'estimacion': pred, 'DE': np.sqrt(var), 'suma_pesos': lam.sum()})
pd.DataFrame(filas).round(3)

In [ ]:
gx = gy = np.linspace(0, 5, 45)
XX, YY = np.meshgrid(gx, gy)
pred = np.empty_like(XX); sd = np.empty_like(XX)
for idx in np.ndindex(XX.shape):
    pred[idx], va, _ = ok(coords, z, [XX[idx], YY[idx]])
    sd[idx] = np.sqrt(va)
fig, axes = plt.subplots(1,2,figsize=(10.5,4.4),constrained_layout=True)
for ax, campo, titulo in [(axes[0],pred,'Estimación'),(axes[1],sd,'DE de kriging')]:
    im=ax.contourf(XX,YY,campo,levels=18,cmap='viridis')
    ax.scatter(coords[:,0],coords[:,1],c='white',edgecolor='black')
    ax.set(xlabel='Este',ylabel='Norte',title=titulo,aspect='equal'); fig.colorbar(im,ax=ax)
plt.show()

## Para interpretar

Una grilla más fina no agrega información. Compare escenarios, retire una observación por vez y calcule errores: la varianza de kriging es condicional al modelo y no reemplaza la validación.